In [10]:
import os, math, json, time, random
from collections import defaultdict
import dotenv
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from openai import OpenAI
from googleapiclient.discovery import build

In [11]:
dotenv.load_dotenv()

True

In [12]:
# OpenAI Token eingeben
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [13]:
# Setting für YouTube
# YouTube Video ID
VIDEO_ID = "6dWYxKW5rY4"

YOUTUBE_API_KEY = os.getenv("YOUTUBE_DATA_API_KEY")

# Maximale Commentzahl (Weil dies ein Demo...)
MAX_PAGES = 5           # 1 Page = 100; Also max. 500 Kommentare
ORDER = "relevance"     # Ranking der Einträge; Ranking nach Zeit ("time") auch möglich 


# YouTube Comments zu einem Video erhalten

In [14]:

def fetch_comments(video_id: str, api_key: str, max_pages: int = 5, order: str = "relevance"):
    yt = build("youtube", "v3", developerKey=api_key, static_discovery=False)
    comments = []
    page_token = None
    pages = 0
    while pages < max_pages:
        resp = yt.commentThreads().list(
            part="snippet",
            videoId=video_id,
            maxResults=100,
            pageToken=page_token,
            textFormat="plainText",
            order=order
        ).execute()
        for item in resp.get("items", []):
            top = item["snippet"]["topLevelComment"]["snippet"]
            text = (top.get("textDisplay") or top.get("textOriginal") or "").strip()
            if not text:
                continue
            comments.append({
                "comment_id": item["id"],
                "text": text,
                "likeCount": int(top.get("likeCount", 0)),
                "publishedAt": top.get("publishedAt", ""),
                "author": top.get("authorDisplayName", "")
            })
        page_token = resp.get("nextPageToken")
        pages += 1
        if not page_token:
            break
    return comments

comments = fetch_comments(VIDEO_ID, YOUTUBE_API_KEY, max_pages=MAX_PAGES, order=ORDER)
len(comments), comments[:3]

(194,
 [{'comment_id': 'UgzVWa_cYdI_xkD8mCx4AaABAg',
   'text': 'Wir schaffen uns ab.',
   'likeCount': 243,
   'publishedAt': '2025-09-05T12:17:58Z',
   'author': '@xwalleex'},
  {'comment_id': 'Ugw8bPkjruq7dOuNMPh4AaABAg',
   'text': 'Merz - Verschwinde!',
   'likeCount': 149,
   'publishedAt': '2025-09-05T12:23:37Z',
   'author': '@Trunksliderman'},
  {'comment_id': 'UgxWSqIXuOb7XSIXLd94AaABAg',
   'text': 'Super geil gemacht das Video 👍Nicht aufgeben Afd! Ich glaube es ist bald geschafft 😊',
   'likeCount': 47,
   'publishedAt': '2025-09-05T12:46:10Z',
   'author': '@K.Franz1963'}])

# Embedding mit dem Modell "text-embedding-3-small"

In [15]:

texts = [c["text"] for c in comments]

# small = Quick & Billig...
EMBED_MODEL = "text-embedding-3-small"

emb = client.embeddings.create(
    model=EMBED_MODEL,
    input=texts
)

print(emb.data[0])

X = np.array([d.embedding for d in emb.data], dtype="float32")
X.shape


Embedding(embedding=[0.010507420636713505, 0.059532757848501205, -0.01390212494879961, 0.003196819918230176, -0.028049517422914505, -0.01236363872885704, -0.018294617533683777, 0.04053578898310661, -0.014504142105579376, 0.03331158682703972, -0.0033417497761547565, -0.030435286462306976, -0.04548570141196251, 0.0034699570387601852, 0.0367899052798748, -0.0012437496334314346, 0.01889663375914097, 0.0010388967348262668, -0.017202068120241165, 0.034426432102918625, 0.026265762746334076, -0.016031481325626373, 0.0006410362548194826, -0.010892041958868504, -0.021650303155183792, -0.03197377175092697, 0.044326264411211014, -0.003901959862560034, -0.00643823342397809, -0.025619152933359146, 0.017090583220124245, -0.02575293369591236, 0.012642350047826767, 0.019063860177993774, 0.016889911144971848, 0.05213018134236336, -0.04744783043861389, 0.006092631258070469, -0.0682731494307518, 0.010211986489593983, 0.022776296362280846, -0.02010066621005535, 0.06394754350185394, 0.007112715393304825, 0.

(194, 1536)

In [16]:
N = len(texts)
k = max(2, min(12, int(math.sqrt(max(N,1)))))

km = KMeans(n_clusters=k, n_init="auto", random_state=0)
labels = km.fit_predict(X)

df = pd.DataFrame(comments)
df["cluster"] = labels
df.head()


/Users/nobu/openaiapi/.venv/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/nobu/openaiapi/.venv/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/nobu/openaiapi/.venv/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


,comment_id,text,likeCount,publishedAt,author,cluster
0,UgzVWa_cYdI_xkD8mCx4AaABAg,Wir schaffen uns ab.,243,2025-09-05T12:17:58Z,@xwalleex,6
1,Ugw8bPkjruq7dOuNMPh4AaABAg,Merz - Verschwinde!,149,2025-09-05T12:23:37Z,@Trunksliderman,7
2,UgxWSqIXuOb7XSIXLd94AaABAg,Super geil gemacht das Video 👍Nicht aufgeben A...,47,2025-09-05T12:46:10Z,@K.Franz1963,5
3,UgwNQ_gg9oo8Bp5lZgd4AaABAg,"Darf man sich eigentlich wünschen, dass Verant...",56,2025-09-05T12:26:14Z,@christopherreichelt3303,9
4,Ugw7e5GR0WWidimIFgB4AaABAg,"Wir brauchen 51%, dafür zählt jede einzelne St...",152,2025-09-05T12:27:02Z,@lb-ns4kj,11


In [17]:
def pick_representatives(rows: pd.DataFrame, topn: int = 12):
    # Zahl des Likes, Länge und Verdopplung berücksichtigen
    rows = rows.copy()
    rows["len"] = rows["text"].str.len()
    rows = rows[(rows["len"]>=5) & (rows["len"]<=300)]
    rows = rows.sort_values(["likeCount", "len"], ascending=[False, True])
    uniq, seen = [], set()
    for t in rows["text"].tolist():
        key = t.strip().lower()
        if key not in seen:
            uniq.append(t.strip())
            seen.add(key)
        if len(uniq) >= topn:
            break
    return uniq

def label_cluster(text_samples, temperature=0.2, model="gpt-4o-mini"):
    system = "Du bist ein Assistent für Forschungsdaten-Analyse. Du erstellst kurze und verständliche Labels und Zusammenfassungen zu einem Cluster auf Deutsch."
    user = (
        "In Folgenden sind YouTube-Kommentare mit ähnlichen Themen"
        "Basierend auf der gemeinsamen Aussagen von dem Cluster, verfasst ein passendes Label und eine Zusammenfassung im folgenden JSON-Format:\n"
        "{\n  \"label\": Kurz und verständlich (in 25 Zeichen),\n"
        "  \"summary\": Zusammenfassung in 2-3 Sätzen,\n"
        "  \"keywords\": Array aus 3-6 Begriffe\n}\n\n"
        "Kommentare:\n- " + "\n- ".join(text_samples)
    )
    resp = client.chat.completions.create(
        model=model,
        temperature=temperature,
        messages=[
            { "role": "system", "content": system },
            { "role": "user", "content": user }
        ],
        response_format={"type":"json_object"}
    )
    return json.loads(resp.choices[0].message.content)

cluster_summaries = {}
for cid in sorted(df["cluster"].unique()):
    reps = pick_representatives(df[df["cluster"]==cid], topn=12)
    try:
        out = label_cluster(reps)
        cluster_summaries[cid] = out
    except Exception as e:
        cluster_summaries[cid] = {"label":"(text generation error)","summary":str(e),"keywords":[]}

cluster_summaries


{np.int32(0): {'label': 'Macht und Einfluss in DE',
  'summary': 'Die Kommentare thematisieren das Gefühl der Ohnmacht des Volkes in Deutschland und hinterfragen, wer tatsächlich die Kontrolle hat. Es wird eine kritische Haltung gegenüber politischen Entscheidungen und dem Einfluss von NGOs geäußert.',
  'keywords': ['Macht', 'Einfluss', 'Volk', 'Politik', 'NGOs']},
 np.int32(1): {'label': 'AfD als Lösung für Deutschland',
  'summary': 'Die Kommentare betonen die Notwendigkeit der AfD als politische Kraft in Deutschland. Viele Nutzer fordern Neuwahlen und sehen die AfD als einzige Alternative zu den etablierten Parteien. Es wird ein starkes Bedürfnis nach Veränderung und einer Rückkehr zu Ordnung und Sicherheit ausgedrückt.',
  'keywords': ['AfD',
   'Neuwahlen',
   'Deutschland',
   'Altparteien',
   'Ordnung',
   'Sicherheit']},
 np.int32(2): {'label': 'Emotionale Reaktionen auf Gewalt',
  'summary': 'Die Kommentare drücken starke Emotionen und Besorgnis über die aktuelle gesellschaf

In [18]:

records = []
for cid in sorted(df['cluster'].unique()):
    info = cluster_summaries.get(cid, {})
    members = df[df['cluster']==cid]
    records.append({
        "cluster_id": int(cid),
        "size": int(len(members)),
        "label": info.get("label",""),
        "summary": info.get("summary",""),
        "keywords": ", ".join(info.get("keywords", []))
    })
report = pd.DataFrame(records).sort_values("size", ascending=False).reset_index(drop=True)
display(report)

# Speichern
os.makedirs("outputs", exist_ok=True)
report_path = "outputs/cluster_report_A.csv"
comments_path = "outputs/comments_A.csv"
report.to_csv(report_path, index=False)
df.to_csv(comments_path, index=False)

report_path, comments_path

,cluster_id,size,label,summary,keywords
0,1,52,AfD als Lösung für Deutschland,Die Kommentare betonen die Notwendigkeit der A...,"AfD, Neuwahlen, Deutschland, Altparteien, Ordn..."
1,6,50,Hoffnung und Frustration,Die Kommentare spiegeln eine Mischung aus Hoff...,"Hoffnung, Frustration, Identität, Migration, G..."
2,8,16,Liebesbekundungen,Die Kommentare drücken eine starke Zuneigung u...,"Liebe, Unterstützung, Deutschland, Emojis, Pos..."
3,2,15,Emotionale Reaktionen auf Gewalt,Die Kommentare drücken starke Emotionen und Be...,"Angst, Gewalt, AfD, Emotionen, Veränderung, Re..."
4,5,14,Positive Unterstützung für Afd,Die Kommentare zeigen eine starke Unterstützun...,"Afd, Unterstützung, Video, Patrioten, Qualität"
5,9,12,Forderung nach Gerechtigkeit,Die Kommentare drücken eine starke Forderung n...,"Gericht, Verantwortung, Strafe, Anklage, Gerec..."
6,7,10,Kritik an Merz,Die Kommentare drücken eine starke Ablehnung g...,"Merz, Kritik, Reformen, Korruption, Merkel"
7,11,10,Wahlaufrufe und Kritik,In den Kommentaren wird ein starkes Plädoyer f...,"AfD, Wahlen, Neuwahlen, Politik, Kritik, Stimmen"
8,0,5,Macht und Einfluss in DE,Die Kommentare thematisieren das Gefühl der Oh...,"Macht, Einfluss, Volk, Politik, NGOs"
9,4,4,YouTube-Kommentare,Die Kommentare beziehen sich auf verschiedene ...,"YouTube, Kommentare, Meinungen, Diskussion, In..."


('outputs/cluster_report_A.csv', 'outputs/comments_A.csv')